# Fine-tune ViT5-base + LoRA — Vietnamese news summarization

Companion notebook for **TICKET-005** (Phase 4 of the roadmap). Designed to run on a free Colab/Kaggle T4 GPU.

Inputs:
- A labeled JSONL dataset produced by `make label-export NAME=v1` (writes `data/datasets/v1/{train,val,test}.jsonl`).
- The repo cloned at `/content/vn-news-summarizer` (Colab) or `/kaggle/working/vn-news-summarizer` (Kaggle).

Outputs:
- `models/vit5-news-v1/` — best LoRA checkpoint + tokenizer config (push to a HF repo or download via the UI to use locally).
- `mlruns/` — MLflow tracking dir with ROUGE / loss curves.

End-to-end on ~1 000 articles: ~30 min wall clock with `num_train_epochs=4`, `batch=4`, `grad_accum=4`, `fp16`.

## 0. Pin dependencies and verify GPU

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q \
    'transformers>=4.46,<5.0' \
    'datasets>=3.1' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'rouge-score>=0.1.2' \
    'sentencepiece>=0.2' \
    'mlflow>=2.17' \
    'PyYAML>=6.0' \
    'loguru>=0.7'

## 1. Clone the repo and the dataset

Replace `<your-fork>` with your repo. The dataset directory is the one written by `make label-export` — bring your own (Drive, Kaggle dataset, or `gsutil cp` from a GCS bucket).

In [ ]:
import os
REPO = 'https://github.com/khangnh22ds/vn-news-summarizer.git'
if not os.path.exists('vn-news-summarizer'):
    !git clone {REPO}
%cd vn-news-summarizer

In [ ]:
# Option A — Mount Google Drive and copy the dataset over.
from google.colab import drive  # type: ignore
drive.mount('/content/drive')
!mkdir -p data/datasets/v1
!cp /content/drive/MyDrive/vn-news/datasets/v1/*.jsonl data/datasets/v1/
!ls -lh data/datasets/v1/

## 2. Make the package importable (uv-style src layout)

In [ ]:
import sys, pathlib
for p in [
    'packages/common/src',
    'packages/training/src',
    'packages/inference/src',
    'scripts',
]:
    sys.path.insert(0, str(pathlib.Path.cwd() / p))
from vn_news_training import load_finetune_config
cfg = load_finetune_config('configs/training/vit5_base_v1.yaml')
cfg

## 3. Run the fine-tune

Either via the Python API (recommended; gives you the trainer object back) or via the CLI.

In [ ]:
from vn_news_training.finetune import run_finetune
result = run_finetune('configs/training/vit5_base_v1.yaml')
result

## 4. Quick eval against the extractive baselines

In [ ]:
!uv run --no-sync python scripts/run_eval.py --baseline lexrank --dataset v1 --no-mlflow --split test
!uv run --no-sync python scripts/run_eval.py --baseline textrank --dataset v1 --no-mlflow --split test
!uv run --no-sync python scripts/run_eval.py --baseline vit5 --model-path models/vit5-news-v1 --dataset v1 --no-mlflow --split test

## 5. Save the checkpoint back to Drive

Compress + copy back so a future Colab session (or your laptop) can pick it up.

In [ ]:
!tar -czvf vit5-news-v1.tar.gz models/vit5-news-v1
!cp vit5-news-v1.tar.gz /content/drive/MyDrive/vn-news/checkpoints/
!ls -lh /content/drive/MyDrive/vn-news/checkpoints/